# Shortpick v3 R14 execution-efficiency analysis

## TL;DR

The deterministic execution snapshot reproduces its saved account ledger exactly. None of the eight bounded variants clears the predeclared replacement gate, so the active strategy set remains unchanged.

## Context, method, and assumptions

This notebook reads the compact governed experiment artifact rather than rebuilding market data. The experiment replays one frozen input snapshot, changes only replacement quality or deployment sizing, and compares nine frontier metrics against both the reproducible baseline and the legacy R14 display contract. One open replacement position is excluded from closed-trade P&L attribution. Historical replay is not forward evidence.

In [1]:
import json
from pathlib import Path

import pandas as pd

ROOT = Path.cwd().parents[1] if Path.cwd().name == 'analysis' else Path.cwd()
ARTIFACT = ROOT / 'docs/contracts/SHORTPICK_V3_R14_EXECUTION_EFFICIENCY_EXPERIMENT_2026-07-15.json'
data = json.loads(ARTIFACT.read_text(encoding='utf-8'))
assert data['status'] == 'completed'
assert data['source_execution_snapshot']['exact_replay'] is True
assert data['accepted_candidate_ids'] == []
data['decision']

'retain_r14_no_execution_efficiency_candidate_cleared_gate'

## Candidate comparison

Return-only improvements are insufficient. A candidate must preserve every governed metric and deliver a material breakthrough or one fewer losing month.

In [2]:
labels = {
    data['variants'][0]['config_id']: 'Reproducible baseline',
    'r14_execution_no_affordable_replacement_research_v1': 'No replacement',
    'r14_execution_replacement_rank4_only_research_v1': 'Rank4 only',
    'r14_execution_replacement_score_gap005_research_v1': 'Score gap 0.05',
    'r14_execution_replacement_fill090_research_v1': 'Minimum fill 90%',
    'r14_execution_deployment_cap_0p07_research_v1': '14 tranches / 7%',
    'r14_execution_deployment_cap_0p075_research_v1': '13 tranches / 7.5%',
    'r14_execution_deployment_cap_0p08_research_v1': '12 tranches / 8%',
    'r14_execution_replacement_rank4_only_deployment070_research_v1': 'Rank4 only + 7%',
}
baseline = data['variants'][0]['summary']
rows = []
for variant in data['variants']:
    summary = variant['summary']
    rows.append({
        'candidate': labels[variant['config_id']],
        'total_return_pct': summary['total_return'] * 100,
        'return_delta_pp': (summary['total_return'] - baseline['total_return']) * 100,
        'max_drawdown_pct': summary['max_drawdown'] * 100,
        'negative_months': summary['negative_month_count'],
        'worst_month_pct': summary['worst_monthly_return'] * 100,
        'skipped_order_pct': summary['skipped_order_rate'] * 100,
        'mean_invested_pct': summary['mean_invested_ratio'] * 100,
        'accepted': variant['config_id'] in data['accepted_candidate_ids'],
    })
candidate_table = pd.DataFrame(rows)
candidate_table.round(3)

,candidate,total_return_pct,return_delta_pp,max_drawdown_pct,negative_months,worst_month_pct,skipped_order_pct,mean_invested_pct,accepted
0,Reproducible baseline,332.100,0.000,-6.885,2,-1.413,15.145,69.197,False
1,No replacement,325.210,-6.891,-7.349,2,-1.418,20.954,68.314,False
2,Rank4 only,333.679,1.578,-6.878,2,-1.418,17.531,68.912,False
3,Score gap 0.05,331.877,-0.224,-6.866,2,-1.413,16.494,68.859,False
4,Minimum fill 90%,331.927,-0.174,-6.847,2,-1.418,18.154,68.953,False
5,14 tranches / 7%,334.123,2.023,-7.089,4,-1.715,15.768,70.860,False
6,13 tranches / 7.5%,324.181,-7.920,-7.734,5,-1.739,16.909,72.820,False
7,12 tranches / 8%,328.178,-3.922,-8.290,5,-2.413,17.842,74.869,False
8,Rank4 only + 7%,332.072,-0.028,-7.047,4,-2.169,18.257,70.622,False


In [3]:
assert not candidate_table['accepted'].any()
assert candidate_table.loc[candidate_table['candidate'] == 'No replacement', 'return_delta_pp'].iloc[0] < -6.8
assert candidate_table.loc[candidate_table['candidate'] == '14 tranches / 7%', 'negative_months'].iloc[0] == 4
candidate_table.sort_values('return_delta_pp', ascending=False)[['candidate', 'return_delta_pp', 'max_drawdown_pct', 'negative_months', 'skipped_order_pct']].round(3)

,candidate,return_delta_pp,max_drawdown_pct,negative_months,skipped_order_pct
5,14 tranches / 7%,2.023,-7.089,4,15.768
2,Rank4 only,1.578,-6.878,2,17.531
0,Reproducible baseline,0.000,-6.885,2,15.145
8,Rank4 only + 7%,-0.028,-7.047,4,18.257
4,Minimum fill 90%,-0.174,-6.847,2,18.154
3,Score gap 0.05,-0.224,-6.866,2,16.494
7,12 tranches / 8%,-3.922,-8.290,5,17.842
1,No replacement,-6.891,-7.349,2,20.954
6,13 tranches / 7.5%,-7.920,-7.734,5,16.909


## Replacement attribution

The replacement mechanism is useful in aggregate, but Rank5 replacements are the weak segment. This is an exploratory diagnostic, not permission to hard-code a post-hoc Rank5 exclusion.

In [4]:
replacement = data['baseline_diagnostics']['replacement_attribution']
replacement_table = pd.DataFrame([
    {'segment': 'All closed replacements', 'closed_count': replacement['closed_sell_count'], 'win_rate_pct': replacement['profitable_closed_rate'] * 100, 'pnl_cny': replacement['closed_total_pnl_cny'], 'mean_return_pct': replacement['closed_mean_return'] * 100, 'median_return_pct': replacement['closed_median_return'] * 100},
    *[{'segment': f'Inventory rank {rank}', 'closed_count': row['closed_count'], 'win_rate_pct': row['profitable_rate'] * 100, 'pnl_cny': row['total_pnl_cny'], 'mean_return_pct': row['mean_return'] * 100, 'median_return_pct': row['median_return'] * 100} for rank, row in replacement['by_inventory_rank'].items()],
])
assert replacement['buy_count'] == replacement['closed_sell_count'] + replacement['open_at_end_count']
replacement_table.round(3)

,segment,closed_count,win_rate_pct,pnl_cny,mean_return_pct,median_return_pct
0,All closed replacements,57,50.877,3464.345,-0.667,0.732
1,Inventory rank 4,35,48.571,4984.619,0.124,-0.449
2,Inventory rank 5,22,54.545,-1520.274,-1.925,1.245


## Cash deployment

The account carries meaningful cash, but the bounded sizing tests show that this is not automatically unused alpha. Higher deployment worsens drawdown and monthly stability.

In [5]:
cash = data['baseline_diagnostics']['cash_deployment']
pd.Series({
    'observed_days': cash['day_count'],
    'mean_cash_pct': cash['mean_cash_ratio'] * 100,
    'median_cash_pct': cash['median_cash_ratio'] * 100,
    'days_cash_ge_40pct': cash['cash_ratio_at_least_40pct_day_count'],
    'days_cash_ge_50pct': cash['cash_ratio_at_least_50pct_day_count'],
}).round(3)

observed_days         675.000
mean_cash_pct          30.803
median_cash_pct        28.703
days_cash_ge_40pct    204.000
days_cash_ge_50pct    134.000
dtype: float64

## Takeaways

1. Keep R14 and do not change the active frontend/backend strategy set.
2. Preserve affordable replacement: removing it materially reduces return and execution coverage.
3. Do not increase deployment simply because average cash is high; tested caps trade stability for small or negative return changes.
4. If another round is run, predeclare a PIT-safe Rank5 replacement-quality feature and require it to replace R14 one-for-one.